# End-to-End Sales Forecasting & Demand Intelligence

This notebook runs the full project pipeline on the Kaggle Superstore `train.csv`. Put that file beside this notebook (or in the repository root) first. Results are computed from the actual input data; no sample values are fabricated.

**Reproducibility:** run `pip install -r requirements.txt`, then execute the notebook from top to bottom. Generated tables, figures, and the business summary are saved in `outputs/`. The Video Game Sales supplementary data has no reliable transaction-level join key to the Superstore dataset, so it is not merged into these sales totals.

In [ ]:
from pathlib import Path
import sys
ROOT = Path.cwd()
if not (ROOT / 'analysis.py').exists() and (ROOT.parent / 'analysis.py').exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
from analysis import load_sales, run_pipeline, monthly_series, OUT, CHARTS
import pandas as pd, numpy as np, json
import matplotlib.pyplot as plt
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.tsa.stattools import adfuller
df = load_sales(ROOT / 'train.csv')
print(f'{len(df):,} transaction rows from {df["Order Date"].min().date()} through {df["Order Date"].max().date()}')

## Task 1 — Loading, feature engineering, and exploration
The loader parses dates, creates year/month/week/day/quarter/season features, drops rows without usable order date or sales, and keeps missing ship dates visible for quality reporting. A merge with video game sales is not defensible because the sources do not share product/customer/date identifiers. Daily, weekly, and monthly totals are created by the pipeline.

In [ ]:
display(df.head())
display(df.dtypes.to_frame('dtype'))
print('Missing values by column:')
display(df.isna().sum().to_frame('missing'))
print('Duplicate rows:', df.duplicated().sum())
print('Category revenue:')
display(df.groupby('Category')['Sales'].sum().sort_values(ascending=False).to_frame('Total sales'))
print('Region sales by year:')
display(df.groupby(['Region','Year'])['Sales'].sum().unstack())
ship = df.dropna(subset=['Ship Date']).assign(ShippingDays=lambda x: (x['Ship Date']-x['Order Date']).dt.days)
display(ship.groupby('Region')['ShippingDays'].agg(['mean','median','count']))
monthly_by_year = df.groupby(['Year','Month'])['Sales'].sum().unstack()
display(monthly_by_year)

**Interpretation prompts:** identify the top category from the table; compare year-over-year regional sales growth and its variability (growth consistency is not the same as highest revenue); inspect shipping-day means by region; and compare each calendar month's sales across years for recurring seasonality. Do not claim causation from a seasonal pattern alone.

## Task 2 — Trend, decomposition, and stationarity
The pipeline saves monthly trend and additive decomposition plots. Stationarity means a series' average level, variability, and dependence structure are reasonably stable over time. The ADF test's null is a unit root/non-stationarity; a small p-value (typically below 0.05) is evidence against that null. First differencing is tested as a transformed series; SARIMA also uses ordinary and annual seasonal differencing in its documented specification.

In [ ]:
monthly = monthly_series(df)
display(monthly.rename('Sales').to_frame().tail())
if len(monthly) >= 24:
    decomposition = seasonal_decompose(monthly, model='additive', period=12, extrapolate_trend='freq')
    decomposition.plot(); plt.tight_layout(); plt.show()
    residual = decomposition.resid.dropna().abs().nlargest(5)
    print('Largest absolute residual months:', residual)
else:
    print('At least 24 monthly observations are needed for annual decomposition.')
if len(monthly) > 4:
    print('ADF original statistic, p-value:', adfuller(monthly, autolag='AIC')[:2])
    print('ADF first difference statistic, p-value:', adfuller(monthly.diff().dropna(), autolag='AIC')[:2])

**Write 3–4 observations after reviewing the generated plot and test output.** In particular: describe the trend, whether the annual pattern repeats, which months have the largest residuals, and what the ADF p-values imply. The decomposition chart is saved to `outputs/charts/decomposition.png`.

## Task 3 — Compare SARIMA, Prophet, and XGBoost
The shared evaluation is a chronological final-three-month holdout. SARIMA uses order (1,1,1) and annual seasonal order (1,1,1,12): differencing addresses level/trend and seasonal differencing captures annual repetition, while low-order AR/MA terms are a compact baseline. Prophet is fit to `ds`/`y` and includes yearly seasonality; weekly seasonality is disabled because the model input is monthly. XGBoost uses lag 1–3, the trailing three-month mean, month, quarter, and season indicators, then recursively predicts the next months. MAE and RMSE are in sales units; MAPE is calculated only where actual sales are nonzero. The lowest holdout MAE selects the provisional best model.

In [ ]:
results = run_pipeline()
display(results['comparison'])
display(results['forecast'])
print('Selected model by chronological holdout MAE:', results['best'])
print('Future SARIMA intervals, if available, are included in the underlying statsmodels output; see below.')
try:
    from statsmodels.tsa.statespace.sarimax import SARIMAX
    fitted = SARIMAX(monthly, order=(1,1,1), seasonal_order=(1,1,1,12), enforce_stationarity=False, enforce_invertibility=False).fit(disp=False)
    display(fitted.get_forecast(3).summary_frame())
except Exception as exc:
    print('SARIMA confidence interval unavailable:', exc)
display(pd.read_csv(OUT / 'segment_forecasts.csv'))
display(pd.read_csv(OUT / 'weekly_anomalies.csv').query('Anomaly == True').head(20))
display(pd.read_csv(OUT / 'demand_segments.csv'))

Prophet's fitted yearly component is plotted in `outputs/charts/prophet_components.png` (where Prophet succeeds). It is appropriate to discuss monthly/annual seasonality here; weekly effects cannot be identified from monthly aggregated observations. The model comparison figure and CSV are `outputs/charts/model_comparison.png` and `outputs/model_comparison.csv`.

## Task 4 — Category and region forecasts
The pipeline applies the selected holdout winner separately to Furniture, Technology, Office Supplies, West, and East, where sufficient history is available. See `outputs/segment_forecasts.csv` and `outputs/charts/segment_forecasts.png`. Report strongest projected growth after comparing the generated segment forecasts; do not compare raw category totals to regional totals as if they were equivalent segments.

## Task 5 — Weekly anomaly detection
Isolation Forest and a rolling eight-week z-score (threshold > 2 standard deviations) flag unusual weekly totals. The combined table records which detector flags each week. A flagged point is not automatically an error or a promotion effect: check it against holiday, marketing, stockout, and data-quality records. See `outputs/weekly_anomalies.csv` and `outputs/charts/weekly_anomalies.png`.

## Task 6 — Product demand segmentation
Sub-categories are represented by total sales, first-to-last available year sales growth, monthly sales volatility, and average order value. Features are standardized, K-Means is fit, and PCA supplies a 2D visualization. An elbow score table is exported; because the elbow can be ambiguous and the number of subcategories is small, interpret the chosen cluster count as exploratory. Cluster labels are heuristic summaries of relative profiles. Suggested policies: protect service levels for high-volume/stable demand; replenish volatile demand more cautiously and frequently; review declining items before adding stock; validate growth before increasing commitments.

## Task 7 — Interactive dashboard
Run `streamlit run app.py`. The app has Sales Overview, Forecast Explorer, Anomaly Report, and Product Demand Segments pages. It reads generated output tables and charts and will run the pipeline on first launch if `train.csv` exists. To deploy, push the dataset or configure an appropriate private data source, connect this repository in Streamlit Community Cloud, and record the actual live URL in your submission. Do not publish a dataset unless its license and intended use allow it.

## Task 8 — Executive report and limitations
The pipeline writes a data-backed first-draft report to `outputs/summary.md`. Export and edit it into a readable two-page PDF/DOCX after reviewing the figures. Use only observed values. Top anomaly causes cannot be determined from sales alone, and no forecast should be treated as a guarantee.

**Limitations:** the models use historical sales only and omit promotions, prices, stockouts, returns, holidays beyond learned patterns, and supply constraints. Short series can weaken seasonal model reliability. Validate results with operational data and track forecast error over time before using them to set purchase orders.

**Run all tasks and save notebook outputs before submission.** Include your actual `train.csv` only if permitted by the dataset terms. Add generated charts, the final PDF/DOCX report, and deployment links to your submission package.